# E-Commerce Customer Intelligence & Sales Analytics
## Notebook 03 — Feature Engineering & Executive KPIs

**IBM SkillsBuild Data Analytics with AI Internship 2026**

---

### Scope

This notebook builds the complete analytical feature layer and executive KPI stack on top of the cleaned datasets from Notebook 02.

| Produced | Used by |
|----------|---------|
| Time features (year, quarter, month, week, dow, hour) | All subsequent notebooks |
| Invoice-level aggregates | Sales analytics, AOV |
| Executive KPI table | Dashboard, reporting |
| Monthly & yearly KPI tables | Trend analysis |
| Customer summary table | RFM, segmentation, CLV |
| Product summary table | Product intelligence |
| Country summary table | Geographic analytics |

> **Stop condition:** This notebook ends after feature engineering, KPI calculation, and summary table construction.
> RFM, segmentation, cohort analysis, market basket, and modelling are out of scope here.

## 1. Imports & Configuration

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 130)

warnings.filterwarnings(
    'ignore', message='.*data validation.*',
    category=UserWarning, module='openpyxl'
)

DATA_PATH   = '../data/online_retail_II.xlsx'
FIGURES_DIR = '../outputs/figures/'
OUTPUTS_DIR = '../outputs/'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
DOW_ORDER   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

print(f'pandas {pd.__version__}  |  numpy {np.__version__}')

## 2. Helper Functions

In [ ]:
def save_figure(filename: str) -> None:
    path = os.path.join(FIGURES_DIR, filename)
    plt.savefig(path, bbox_inches='tight')
    print(f'Figure saved: {path}')


def classify_stockcode(code: str, description: str) -> str:
    """
    Classify a StockCode into a product_type category.
    Identical to Notebook 02 — kept here for self-contained reproducibility.
    """
    code = str(code).strip().upper() if pd.notna(code) else ''
    desc = str(description).strip().upper() if pd.notna(description) else ''
    if re.match(r'^TEST', code) or 'TEST' in desc:                         return 'TEST'
    if code in ('B', 'ADJUST2') or 'BAD DEBT' in desc:                     return 'BAD_DEBT'
    if code.startswith('GIFT') or 'GIFT VOUCHER' in desc:                  return 'GIFT_VOUCHER'
    if code in ('POST','DOT','C2') or 'POSTAGE' in desc or 'CARRIAGE' in desc: return 'POSTAGE'
    if code in ('BANK CHARGES','BANKCHARGES','AMAZONFEE') \
            or 'BANK CHARGE' in desc or 'AMAZON FEE' in desc or 'FEE' in desc: return 'FEE_OR_CHARGE'
    if code == 'D' or 'DISCOUNT' in desc:                                  return 'DISCOUNT'
    if code == 'S' or 'SAMPLE' in desc:                                    return 'SAMPLE'
    if code in ('M','ADJUST','CRUK') or 'MANUAL' in desc or 'ADJUST' in desc or 'CRUK' in desc:
        return 'MANUAL_ADJUSTMENT'
    if re.match(r'^\d{5}[A-Z]?$', code):                                   return 'MERCHANDISE'
    return 'UNCLASSIFIED_NON_STANDARD'


def assign_transaction_type(row) -> str:
    invoice = str(row['Invoice']).strip().upper()
    qty, price = row['Quantity'], row['Price']
    if invoice.startswith('C'):   return 'CANCELLED_INVOICE'
    if qty < 0:                   return 'RETURN_OR_NEGATIVE_ADJUSTMENT'
    if qty == 0:                  return 'ZERO_QUANTITY'
    if qty > 0:
        if price > 0:  return 'SALE'
        if price == 0: return 'ZERO_PRICE_POSITIVE_QTY'
        if price < 0:  return 'NEGATIVE_PRICE_POSITIVE_QTY'
    return 'OTHER_ADJUSTMENT'


def modal_description(s: pd.Series):
    nn = s.dropna()
    return nn.mode().iloc[0] if len(nn) > 0 else None


print('Helper functions defined.')

## 3. Load Raw Data & Rebuild Cleaned Datasets

Reuses the exact same pipeline as Notebook 02 — no cleaning logic is duplicated beyond what is necessary for a self-contained, reproducible notebook.

In [ ]:
xl_file     = pd.ExcelFile(DATA_PATH, engine='openpyxl')
sheet_names = xl_file.sheet_names
print(f'Sheets: {sheet_names}')

sheets = {}
for name in sheet_names:
    print(f'  Loading "{name}" ...', end=' ')
    df = pd.read_excel(DATA_PATH, sheet_name=name, engine='openpyxl',
                       dtype={'Invoice': str, 'StockCode': str})
    sheets[name] = df
    print(f'{len(df):,} rows')

tagged = []
for name, df in sheets.items():
    tmp = df.copy()
    tmp['_sheet'] = name
    tagged.append(tmp)

raw = pd.concat(tagged, ignore_index=True)
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'], errors='coerce')
RAW_ROW_COUNT = len(raw)

# Exact duplicate removal
dup_mask              = raw.duplicated(keep='first')
n_duplicates          = dup_mask.sum()
cleaned_transactions  = raw[~dup_mask].copy().reset_index(drop=True)

assert len(cleaned_transactions) + n_duplicates == RAW_ROW_COUNT
print(f'\nRaw rows         : {RAW_ROW_COUNT:,}')
print(f'Duplicates removed: {n_duplicates:,}')
print(f'Cleaned rows      : {len(cleaned_transactions):,}')

In [ ]:
# ── Price flags ───────────────────────────────────────────────────────────────
cleaned_transactions['is_zero_price']     = cleaned_transactions['Price'] == 0
cleaned_transactions['is_negative_price'] = cleaned_transactions['Price'] < 0

# ── Classification ────────────────────────────────────────────────────────────
cleaned_transactions['product_type'] = cleaned_transactions.apply(
    lambda r: classify_stockcode(r['StockCode'], r['Description']), axis=1)

cleaned_transactions['transaction_type'] = cleaned_transactions.apply(
    assign_transaction_type, axis=1)

# ── Boolean flags & revenue ───────────────────────────────────────────────────
cleaned_transactions['is_cancelled']         = cleaned_transactions['transaction_type'] == 'CANCELLED_INVOICE'
cleaned_transactions['is_negative_quantity'] = cleaned_transactions['Quantity'] < 0
cleaned_transactions['is_positive_sale']     = cleaned_transactions['transaction_type'] == 'SALE'
cleaned_transactions['revenue']              = cleaned_transactions['Quantity'] * cleaned_transactions['Price']

# ── Canonical description ─────────────────────────────────────────────────────
canonical_desc_map = cleaned_transactions.groupby('StockCode')['Description'].agg(modal_description)
cleaned_transactions['canonical_description'] = cleaned_transactions['StockCode'].map(canonical_desc_map)

# ── Analytical datasets ───────────────────────────────────────────────────────
valid_merchandise_transactions = cleaned_transactions[
    (cleaned_transactions['transaction_type'] == 'SALE') &
    (cleaned_transactions['product_type'] == 'MERCHANDISE')
].copy()

customer_transactions = valid_merchandise_transactions[
    valid_merchandise_transactions['Customer ID'].notna()
].copy()

merchandise_product_transactions = cleaned_transactions[
    cleaned_transactions['product_type'] == 'MERCHANDISE'
].copy()

# Assertions
assert (cleaned_transactions['revenue'].round(8) ==
        (cleaned_transactions['Quantity'] * cleaned_transactions['Price']).round(8)).all()
assert customer_transactions['Customer ID'].notna().all()

print('Datasets ready:')
for lbl, df in [
    ('cleaned_transactions',             cleaned_transactions),
    ('valid_merchandise_transactions',   valid_merchandise_transactions),
    ('customer_transactions',            customer_transactions),
    ('merchandise_product_transactions', merchandise_product_transactions),
]:
    print(f'  {lbl:<45}: {len(df):>10,} rows')

## 4. Section 2 — Data Type Standardization

Verify that all analytical columns have the expected types before any feature engineering.

In [ ]:
for df_name, df in [
    ('cleaned_transactions',           cleaned_transactions),
    ('valid_merchandise_transactions', valid_merchandise_transactions),
    ('customer_transactions',          customer_transactions),
]:
    assert df['Invoice'].dtype == object or pd.api.types.is_string_dtype(df['Invoice']), \
        f"{df_name}: Invoice not string-like"
    assert df['StockCode'].dtype == object or pd.api.types.is_string_dtype(df['StockCode']), \
        f"{df_name}: StockCode not string-like"
    assert pd.api.types.is_datetime64_any_dtype(df['InvoiceDate']), \
        f"{df_name}: InvoiceDate not datetime"
    assert pd.api.types.is_numeric_dtype(df['Quantity']), f"{df_name}: Quantity not numeric"
    assert pd.api.types.is_numeric_dtype(df['Price']),    f"{df_name}: Price not numeric"
    assert pd.api.types.is_numeric_dtype(df['revenue']),  f"{df_name}: revenue not numeric"

print('All dtype assertions passed.')
print('\nColumn dtypes (cleaned_transactions):')
display(cleaned_transactions.dtypes.rename('dtype').reset_index().rename(columns={'index':'column'}))

## 5. Section 3 — Time Features

Derived from `InvoiceDate`. Applied to all four analytical datasets so each is self-contained.

In [ ]:
for df in [cleaned_transactions, valid_merchandise_transactions,
           customer_transactions, merchandise_product_transactions]:
    dt = df['InvoiceDate']
    df['year']             = dt.dt.year
    df['quarter']          = dt.dt.quarter
    df['month']            = dt.dt.month
    df['month_name']       = pd.Categorical(dt.dt.strftime('%B'),
                                            categories=MONTH_ORDER, ordered=True)
    df['year_month']       = dt.dt.to_period('M')
    df['week']             = dt.dt.isocalendar().week.astype('Int64')
    df['day']              = dt.dt.day
    df['day_of_week']      = dt.dt.dayofweek          # 0 = Monday
    df['day_of_week_name'] = pd.Categorical(dt.dt.strftime('%A'),
                                            categories=DOW_ORDER, ordered=True)
    df['hour']             = dt.dt.hour

# Verify chronological sort of year_month
ym = customer_transactions['year_month'].dropna().sort_values()
assert list(ym) == sorted(ym.tolist()), 'year_month does not sort chronologically'

print('Time features added to all datasets.')
print(f"  Date range : {cleaned_transactions['InvoiceDate'].min()} to "
      f"{cleaned_transactions['InvoiceDate'].max()}")
print(f"  Years      : {sorted(cleaned_transactions['year'].dropna().unique().tolist())}")
print(f"  Months     : {cleaned_transactions['year_month'].nunique()} distinct year-months")
print('  Chronological sort assertion: PASSED')

## 6. Section 4 — Transaction-Level & Invoice-Level Features

Each row in `valid_merchandise_transactions` is one line item. A single invoice may contain many rows.

**Invoice-level aggregates** collapse rows to the invoice grain.

In [ ]:
invoice_agg = (
    valid_merchandise_transactions
    .groupby('Invoice')
    .agg(
        invoice_revenue            =('revenue',      'sum'),
        invoice_units              =('Quantity',     'sum'),
        unique_products_per_invoice=('StockCode',    'nunique'),
        invoice_date               =('InvoiceDate',  'first'),
        customer_id                =('Customer ID',  'first'),
        country                    =('Country',      'first'),
    )
    .reset_index()
)

print(f'Invoice-level rows : {len(invoice_agg):,}  (one row per unique merchandise invoice)')
print(f'\nInvoice revenue distribution:')
print(invoice_agg['invoice_revenue'].describe(percentiles=[.25,.5,.75,.90,.95,.99]))

# Sanity check: line-item count > invoice count
assert len(valid_merchandise_transactions) > len(invoice_agg), \
    'Expected more line items than invoices'
print(f'\nLine items : {len(valid_merchandise_transactions):,}')
print(f'Invoices   : {len(invoice_agg):,}')
print(f'Avg lines  : {len(valid_merchandise_transactions)/len(invoice_agg):.1f} per invoice')

## 7. Section 5 — Metric Definitions

All KPIs below are calculated using these precise definitions.

| Metric | Definition | Dataset |
|--------|------------|---------|
| **Gross Merchandise Revenue** | Sum of `revenue` where `transaction_type == SALE` and `product_type == MERCHANDISE` | `valid_merchandise_transactions` |
| **Return / Cancellation Value** | Absolute sum of negative `revenue` on merchandise rows | `merchandise_product_transactions` |
| **Net Merchandise Revenue** | Gross Revenue + Return Value (return value is negative, so this reduces gross) | derived |
| **Orders** | Count of unique `Invoice` values in `valid_merchandise_transactions` — NOT row count | `valid_merchandise_transactions` |
| **Units Sold** | Sum of `Quantity` in `valid_merchandise_transactions` | `valid_merchandise_transactions` |
| **Unique Customers** | Count of unique `Customer ID` values in `customer_transactions` | `customer_transactions` |
| **Average Order Value (AOV)** | Mean of `invoice_revenue` across all merchandise invoices — **gross definition** | `invoice_agg` |
| **Average Items per Order** | Mean of `invoice_units` across all merchandise invoices | `invoice_agg` |
| **Repeat Customer** | A `Customer ID` appearing on more than 1 unique `Invoice` in `customer_transactions` | `customer_transactions` |
| **Repeat Customer Rate** | `Repeat Customers / Total Identified Customers × 100` | derived |
| **Cancellation Invoice Rate** | `Cancellation Invoices / All Unique Invoices × 100` — denominator is the full invoice universe in `cleaned_transactions` | `cleaned_transactions` |

## 8. Section 6 — Executive KPI Table

In [ ]:
# ── Gross merchandise revenue ─────────────────────────────────────────────────
gross_revenue    = valid_merchandise_transactions['revenue'].sum()

# ── Return / cancellation value ───────────────────────────────────────────────
return_value     = merchandise_product_transactions[
    merchandise_product_transactions['revenue'] < 0]['revenue'].sum()    # negative
return_value_abs = abs(return_value)

# ── Net merchandise revenue ───────────────────────────────────────────────────
net_revenue      = gross_revenue + return_value

# ── Orders ───────────────────────────────────────────────────────────────────
n_orders         = valid_merchandise_transactions['Invoice'].nunique()

# ── Units sold ────────────────────────────────────────────────────────────────
units_sold       = valid_merchandise_transactions['Quantity'].sum()

# ── Unique customers ─────────────────────────────────────────────────────────
n_customers      = customer_transactions['Customer ID'].nunique()

# ── Unique products ───────────────────────────────────────────────────────────
n_products       = valid_merchandise_transactions['StockCode'].nunique()

# ── AOV ───────────────────────────────────────────────────────────────────────
aov              = invoice_agg['invoice_revenue'].mean()
avg_items        = invoice_agg['invoice_units'].mean()

# ── Repeat customers ─────────────────────────────────────────────────────────
customer_order_counts = customer_transactions.groupby('Customer ID')['Invoice'].nunique()
repeat_customers      = (customer_order_counts > 1).sum()
total_id_customers    = customer_order_counts.shape[0]
repeat_rate           = repeat_customers / total_id_customers * 100

# ── Cancellation rate ─────────────────────────────────────────────────────────
cancel_invoices   = cleaned_transactions[cleaned_transactions['is_cancelled']]['Invoice'].nunique()
total_invoices    = cleaned_transactions['Invoice'].nunique()
cancel_rate       = cancel_invoices / total_invoices * 100

kpi_table = pd.DataFrame([
    {'KPI': 'Gross Merchandise Revenue',    'Value': f'£{gross_revenue:,.2f}',    'Dataset': 'valid_merchandise_transactions'},
    {'KPI': 'Return / Cancellation Value',  'Value': f'£{return_value_abs:,.2f}', 'Dataset': 'merchandise_product_transactions (negative rows)'},
    {'KPI': 'Net Merchandise Revenue',      'Value': f'£{net_revenue:,.2f}',      'Dataset': 'derived'},
    {'KPI': 'Orders (unique invoices)',      'Value': f'{n_orders:,}',             'Dataset': 'valid_merchandise_transactions'},
    {'KPI': 'Units Sold',                   'Value': f'{units_sold:,}',           'Dataset': 'valid_merchandise_transactions'},
    {'KPI': 'Unique Customers',             'Value': f'{n_customers:,}',          'Dataset': 'customer_transactions'},
    {'KPI': 'Unique Products',              'Value': f'{n_products:,}',           'Dataset': 'valid_merchandise_transactions'},
    {'KPI': 'Average Order Value (gross)',  'Value': f'£{aov:,.2f}',              'Dataset': 'invoice_agg'},
    {'KPI': 'Average Items per Order',      'Value': f'{avg_items:,.1f}',         'Dataset': 'invoice_agg'},
    {'KPI': 'Repeat Customers',             'Value': f'{repeat_customers:,}',     'Dataset': 'customer_transactions'},
    {'KPI': 'Repeat Customer Rate',         'Value': f'{repeat_rate:.2f}%',       'Dataset': 'customer_transactions'},
    {'KPI': 'Cancellation Invoice Rate',    'Value': f'{cancel_rate:.2f}%',       'Dataset': 'cleaned_transactions'},
])

print('EXECUTIVE KPI TABLE')
display(kpi_table)

## 9. Section 7 — Monthly KPI Table

In [ ]:
# ── Revenue from merchandise_product_transactions (includes negative/cancel rows) ──
monthly_merch = (
    merchandise_product_transactions
    .groupby('year_month')
    .agg(
        gross_revenue =('revenue', lambda x: x[x > 0].sum()),
        return_value  =('revenue', lambda x: x[x < 0].sum()),
    )
    .reset_index()
)
monthly_merch['net_revenue']      = monthly_merch['gross_revenue'] + monthly_merch['return_value']
monthly_merch['return_value_abs'] = monthly_merch['return_value'].abs()

# ── Orders & units from valid_merchandise_transactions ───────────────────────
monthly_sales = (
    valid_merchandise_transactions
    .groupby('year_month')
    .agg(orders=('Invoice','nunique'), units=('Quantity','sum'))
    .reset_index()
)

# ── Unique customers from customer_transactions ───────────────────────────────
monthly_cust = (
    customer_transactions
    .groupby('year_month')['Customer ID']
    .nunique()
    .reset_index()
    .rename(columns={'Customer ID': 'unique_customers'})
)

# ── Average order value from invoice_agg ─────────────────────────────────────
invoice_agg['year_month'] = invoice_agg['invoice_date'].dt.to_period('M')
monthly_aov = (
    invoice_agg
    .groupby('year_month')['invoice_revenue']
    .mean()
    .reset_index()
    .rename(columns={'invoice_revenue': 'average_order_value'})
)

monthly_kpis = (
    monthly_merch
    .merge(monthly_sales, on='year_month', how='outer')
    .merge(monthly_cust,  on='year_month', how='outer')
    .merge(monthly_aov,   on='year_month', how='outer')
    .sort_values('year_month')
    .reset_index(drop=True)
)

# ── Month-over-month net revenue growth ───────────────────────────────────────
monthly_kpis['net_revenue_growth_pct'] = monthly_kpis['net_revenue'].pct_change() * 100
# Mask first period (no prior period) and cases where prior period = 0 (undefined growth)
monthly_kpis.loc[monthly_kpis['net_revenue'].shift(1) == 0, 'net_revenue_growth_pct'] = np.nan

print(f'Monthly KPI table: {len(monthly_kpis)} rows  (one per year-month)')
display(monthly_kpis[['year_month','gross_revenue','return_value_abs','net_revenue',
                       'orders','units','unique_customers',
                       'average_order_value','net_revenue_growth_pct']])

## 10. Section 8 — Yearly Comparison

> **Important:** 2009 is represented by only 1 month of data (December 2009). Year-over-year comparisons involving 2009 are not meaningful and are flagged as PARTIAL.

In [ ]:
yearly_merch = (
    merchandise_product_transactions
    .groupby('year')
    .agg(
        gross_revenue=('revenue', lambda x: x[x > 0].sum()),
        return_value =('revenue', lambda x: x[x < 0].sum()),
    )
    .reset_index()
)
yearly_merch['net_revenue']      = yearly_merch['gross_revenue'] + yearly_merch['return_value']
yearly_merch['return_value_abs'] = yearly_merch['return_value'].abs()

yearly_sales = (
    valid_merchandise_transactions
    .groupby('year')
    .agg(orders=('Invoice','nunique'), units=('Quantity','sum'))
    .reset_index()
)

yearly_cust = (
    customer_transactions
    .groupby('year')['Customer ID']
    .nunique()
    .reset_index()
    .rename(columns={'Customer ID': 'unique_customers'})
)

invoice_agg['year'] = invoice_agg['invoice_date'].dt.year
yearly_aov = (
    invoice_agg
    .groupby('year')['invoice_revenue']
    .mean()
    .reset_index()
    .rename(columns={'invoice_revenue': 'average_order_value'})
)

yearly_kpis = (
    yearly_merch
    .merge(yearly_sales, on='year', how='outer')
    .merge(yearly_cust,  on='year', how='outer')
    .merge(yearly_aov,   on='year', how='outer')
    .sort_values('year')
    .reset_index(drop=True)
)

for col in ['gross_revenue', 'net_revenue', 'orders', 'units']:
    yearly_kpis[f'{col}_yoy_pct'] = yearly_kpis[col].pct_change() * 100

# Months per year (for partial-year annotation)
months_per_year = cleaned_transactions.groupby('year')['year_month'].nunique()

yearly_kpis['months_in_data'] = yearly_kpis['year'].map(months_per_year)
yearly_kpis['period_note']    = yearly_kpis['months_in_data'].apply(
    lambda m: 'PARTIAL' if m < 12 else 'FULL'
)

display(yearly_kpis[['year','period_note','months_in_data',
                      'gross_revenue','return_value_abs','net_revenue',
                      'orders','units','unique_customers','average_order_value',
                      'net_revenue_yoy_pct','orders_yoy_pct']])

print('\nPartial-year caveat: 2009 contains only December data.')
print('YoY % for 2010 vs 2009 reflects 12 months vs 1 month — not a valid annual comparison.')

## 11. Section 9 — Customer Summary Table

Built from `customer_transactions` (valid merchandise sales with identified Customer ID).

No RFM scoring is performed here.

In [ ]:
customer_summary = (
    customer_transactions
    .groupby('Customer ID')
    .agg(
        first_purchase_date=('InvoiceDate', 'min'),
        last_purchase_date =('InvoiceDate', 'max'),
        total_orders       =('Invoice',     'nunique'),
        total_units        =('Quantity',    'sum'),
        total_revenue      =('revenue',     'sum'),
        unique_products    =('StockCode',   'nunique'),
    )
    .reset_index()
    .rename(columns={'Customer ID': 'customer_id'})
)

customer_summary['average_order_value']    = (
    customer_summary['total_revenue'] / customer_summary['total_orders']
)
customer_summary['customer_lifetime_days'] = (
    (customer_summary['last_purchase_date'] -
     customer_summary['first_purchase_date']).dt.days
)

print(f'Customers in summary: {len(customer_summary):,}')
print('\nDistribution statistics:')
display(customer_summary[[
    'total_orders','total_units','total_revenue',
    'unique_products','average_order_value','customer_lifetime_days'
]].describe(percentiles=[.25,.5,.75,.90,.95,.99]))

# Reconciliation
cust_rev_sum = customer_summary['total_revenue'].sum()
cust_txn_rev = customer_transactions['revenue'].sum()
discrepancy  = abs(cust_rev_sum - cust_txn_rev)
print(f'\nRevenue reconciliation:')
print(f'  customer_summary total_revenue sum : £{cust_rev_sum:,.2f}')
print(f'  customer_transactions revenue sum  : £{cust_txn_rev:,.2f}')
print(f'  Discrepancy                        : £{discrepancy:,.6f}')
assert discrepancy < 0.01, f'Customer revenue reconciliation failed: £{discrepancy:.4f}'
print('  [PASS]')

In [ ]:
print('Sample customer_summary (first 10 rows, sorted by total_revenue desc):')
display(
    customer_summary.sort_values('total_revenue', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

## 12. Section 10 — Product Summary Table

- **Gross revenue** comes from `valid_merchandise_transactions` (positive SALE rows only).
- **Return value** comes from `merchandise_product_transactions` negative rows.
- Outer join used so products with returns but no forward sales are captured.

In [ ]:
product_gross = (
    valid_merchandise_transactions
    .groupby('StockCode')
    .agg(
        canonical_description=('canonical_description', 'first'),
        total_units_sold     =('Quantity',     'sum'),
        gross_revenue        =('revenue',      'sum'),
        number_of_orders     =('Invoice',      'nunique'),
        number_of_customers  =('Customer ID',  'nunique'),
    )
    .reset_index()
)

product_returns = (
    merchandise_product_transactions[
        merchandise_product_transactions['revenue'] < 0
    ]
    .groupby('StockCode')['revenue']
    .sum()
    .reset_index()
    .rename(columns={'revenue': 'return_value'})
)

product_summary = product_gross.merge(product_returns, on='StockCode', how='outer')
product_summary['return_value']          = product_summary['return_value'].fillna(0)
product_summary['gross_revenue']         = product_summary['gross_revenue'].fillna(0)
product_summary['total_units_sold']      = product_summary['total_units_sold'].fillna(0)
product_summary['number_of_orders']      = product_summary['number_of_orders'].fillna(0)
product_summary['number_of_customers']   = product_summary['number_of_customers'].fillna(0)
product_summary['canonical_description'] = product_summary['canonical_description'].fillna(
    product_summary['StockCode'].map(canonical_desc_map)
)
product_summary['net_revenue'] = product_summary['gross_revenue'] + product_summary['return_value']

# Reconciliation
discrepancy_gross = abs(product_summary['gross_revenue'].sum() -
                        valid_merchandise_transactions['revenue'].sum())
discrepancy_net   = abs(product_summary['net_revenue'].sum() -
                        merchandise_product_transactions['revenue'].sum())

print(f'Products in summary: {len(product_summary):,}')
print(f'  Gross discrepancy: £{discrepancy_gross:,.4f}')
print(f'  Net  discrepancy : £{discrepancy_net:,.4f}')
assert discrepancy_gross < 0.01
assert discrepancy_net   < 0.01
print('  [PASS] Product revenue reconciles.')

print('\nTop 10 products by gross revenue:')
display(
    product_summary.sort_values('gross_revenue', ascending=False)
    .head(10)
    .reset_index(drop=True)
    [['StockCode','canonical_description','total_units_sold',
      'gross_revenue','return_value','net_revenue','number_of_orders']]
)

## 13. Section 11 — Geographic Summary Table

- Transaction-volume metrics (gross revenue, orders, units) use `valid_merchandise_transactions`.
- Customer-level counts use `customer_transactions` (identified customers only).
- Return values use `merchandise_product_transactions` negative rows.

In [ ]:
country_gross = (
    valid_merchandise_transactions
    .groupby('Country')
    .agg(gross_revenue=('revenue','sum'),
         orders=('Invoice','nunique'),
         units_sold=('Quantity','sum'))
    .reset_index()
)

country_cust = (
    customer_transactions
    .groupby('Country')['Customer ID']
    .nunique()
    .reset_index()
    .rename(columns={'Customer ID': 'unique_customers'})
)

country_returns = (
    merchandise_product_transactions[
        merchandise_product_transactions['revenue'] < 0
    ]
    .groupby('Country')['revenue']
    .sum()
    .reset_index()
    .rename(columns={'revenue': 'return_value'})
)

country_summary = (
    country_gross
    .merge(country_cust,    on='Country', how='left')
    .merge(country_returns, on='Country', how='left')
)
country_summary['return_value']       = country_summary['return_value'].fillna(0)
country_summary['unique_customers']   = country_summary['unique_customers'].fillna(0).astype(int)
country_summary['net_revenue']        = country_summary['gross_revenue'] + country_summary['return_value']
country_summary['average_order_value'] = country_summary['gross_revenue'] / country_summary['orders']
country_summary['revenue_per_customer'] = np.where(
    country_summary['unique_customers'] > 0,
    country_summary['gross_revenue'] / country_summary['unique_customers'],
    np.nan
)
country_summary = country_summary.sort_values('gross_revenue', ascending=False).reset_index(drop=True)

# Reconciliation
disc_country = abs(country_summary['gross_revenue'].sum() -
                   valid_merchandise_transactions['revenue'].sum())
assert disc_country < 0.01, f'Country reconciliation failed: £{disc_country:.4f}'
print(f'Countries: {len(country_summary)}')
print('[PASS] Country gross revenue reconciles.\n')
display(country_summary)

## 14. Section 12 — Visualizations

### 14.1 KPI Summary Block

In [ ]:
kpi_display = [
    ('Gross Revenue',     f'£{gross_revenue/1e6:.2f}M'),
    ('Return Value',      f'£{return_value_abs/1e3:.0f}K'),
    ('Net Revenue',       f'£{net_revenue/1e6:.2f}M'),
    ('Orders',            f'{n_orders:,}'),
    ('Units Sold',        f'{units_sold/1e6:.1f}M'),
    ('Unique Customers',  f'{n_customers:,}'),
    ('Unique Products',   f'{n_products:,}'),
    ('Avg Order Value',   f'£{aov:,.2f}'),
    ('Repeat Rate',       f'{repeat_rate:.1f}%'),
    ('Cancel Rate',       f'{cancel_rate:.1f}%'),
]

cols_k, rows_k = 5, 2
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')
ax.set_facecolor('#f7f8fa')
fig.patch.set_facecolor('#f7f8fa')

for i, (label, val) in enumerate(kpi_display):
    col_i = i % cols_k
    row_i = i // cols_k
    x = col_i / cols_k + 0.04
    y = 0.88 - row_i * 0.46
    ax.text(x, y,        label, transform=ax.transAxes, fontsize=10,
            color='#57606a', va='top', fontweight='normal')
    ax.text(x, y - 0.22, val,   transform=ax.transAxes, fontsize=16,
            color='#1f2328', va='top', fontweight='bold')

ax.set_title('Executive KPI Summary  |  Dec 2009 – Dec 2011',
             fontsize=13, fontweight='bold', pad=12, color='#1f2328')
plt.tight_layout()
save_figure('09_kpi_summary.png')
plt.show()

### 14.2 Monthly Gross vs Net Revenue

In [ ]:
ym_labels = [str(p) for p in monthly_kpis['year_month']]
x_pos = range(len(ym_labels))
fmt_gbp = mticker.FuncFormatter(lambda v, _: f'£{v/1e6:.1f}M' if abs(v) >= 1e6 else f'£{v:,.0f}')

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x_pos, monthly_kpis['gross_revenue'], label='Gross Revenue',
       alpha=0.85, color=sns.color_palette('muted')[0], edgecolor='white')
ax.bar(x_pos, monthly_kpis['net_revenue'],   label='Net Revenue',
       alpha=0.85, color=sns.color_palette('muted')[2], edgecolor='white')
ax.set_xticks(list(x_pos))
ax.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(fmt_gbp)
ax.set_title('Monthly Gross vs Net Merchandise Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.legend()
plt.tight_layout()
save_figure('10_monthly_gross_vs_net_revenue.png')
plt.show()

### 14.3 Monthly Order Count

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(list(x_pos), monthly_kpis['orders'].fillna(0),
       color=sns.color_palette('muted')[1], edgecolor='white')
ax.set_xticks(list(x_pos))
ax.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title('Monthly Order Count (Unique Merchandise Invoices)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Orders')
plt.tight_layout()
save_figure('11_monthly_orders.png')
plt.show()

### 14.4 Monthly Unique Customers

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(list(x_pos), monthly_kpis['unique_customers'].fillna(0),
       color=sns.color_palette('muted')[3], edgecolor='white')
ax.set_xticks(list(x_pos))
ax.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title('Monthly Unique Customers (Identified, Merchandise Purchases)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Unique Customers')
plt.tight_layout()
save_figure('12_monthly_unique_customers.png')
plt.show()

### 14.5 Monthly Average Order Value

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(list(x_pos), monthly_kpis['average_order_value'].fillna(np.nan),
        marker='o', linewidth=2, color=sns.color_palette('muted')[4], markersize=4)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(ym_labels, rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax.set_title('Monthly Average Order Value (Gross Merchandise)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('AOV (£)')
plt.tight_layout()
save_figure('13_monthly_aov.png')
plt.show()

### 14.6 Yearly Net Revenue Comparison

In [ ]:
yr_labels = [str(y) for y in yearly_kpis['year']]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(yr_labels, yearly_kpis['net_revenue'],
              color=sns.color_palette('muted', n_colors=len(yr_labels)),
              edgecolor='white')
for bar, row in zip(bars, yearly_kpis.itertuples()):
    note = f' ({row.period_note})' if row.period_note == 'PARTIAL' else ''
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + yearly_kpis['net_revenue'].max() * 0.01,
            f'£{row.net_revenue/1e6:.2f}M{note}',
            ha='center', va='bottom', fontsize=10)
ax.yaxis.set_major_formatter(fmt_gbp)
ax.set_title('Yearly Net Merchandise Revenue', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Net Revenue (£)')
ax.text(0.02, 0.97,
        'Note: 2009 = December only (PARTIAL)',
        transform=ax.transAxes, fontsize=8, color='#57606a',
        va='top', style='italic')
plt.tight_layout()
save_figure('14_yearly_net_revenue.png')
plt.show()

### 14.7 Customer Revenue Distribution

In [ ]:
p99 = customer_summary['total_revenue'].quantile(0.99)
clipped_cust = customer_summary['total_revenue'][
    customer_summary['total_revenue'] <= p99
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(clipped_cust, bins=60, color=sns.color_palette('muted')[0],
        edgecolor='white', linewidth=0.4)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title(
    f'Customer Total Revenue Distribution  (capped at 99th pct = £{p99:,.0f})',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Total Revenue per Customer (£)')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
save_figure('15_customer_revenue_distribution.png')
plt.show()
print(f'Customers plotted (≤ p99): {len(clipped_cust):,} of {len(customer_summary):,}')

### 14.8 Order Value Distribution

In [ ]:
p99_inv = invoice_agg['invoice_revenue'].quantile(0.99)
clipped_inv = invoice_agg['invoice_revenue'][
    invoice_agg['invoice_revenue'] <= p99_inv
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(clipped_inv, bins=60, color=sns.color_palette('muted')[1],
        edgecolor='white', linewidth=0.4)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.set_title(
    f'Order Value Distribution  (capped at 99th pct = £{p99_inv:,.0f})',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Order Value (£)')
ax.set_ylabel('Number of Orders')
plt.tight_layout()
save_figure('16_order_value_distribution.png')
plt.show()
print(f'Orders plotted (≤ p99): {len(clipped_inv):,} of {len(invoice_agg):,}')

### 14.9 Top 10 Countries by Net Revenue

In [ ]:
top10_c = country_summary.head(10)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top10_c['Country'][::-1], top10_c['net_revenue'][::-1],
        color=sns.color_palette('muted', n_colors=10)[::-1], edgecolor='white')
ax.xaxis.set_major_formatter(fmt_gbp)
ax.set_title('Top 10 Countries by Net Merchandise Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Net Revenue (£)')
ax.set_ylabel('Country')
plt.tight_layout()
save_figure('17_top10_countries_net_revenue.png')
plt.show()

## 15. Section 13 — Reconciliation & Validation

In [ ]:
print('Running reconciliation suite...')
checks = []

def chk(label, condition, detail=''):
    status = 'PASS' if condition else 'FAIL'
    checks.append((label, status))
    print(f'  [{status}] {label}' + (f'  ({detail})' if detail else ''))
    assert condition, f'Reconciliation failed: {label}'

# 1. Revenue formula
chk('revenue == Quantity * Price on all rows',
    (cleaned_transactions['revenue'].round(8) ==
     (cleaned_transactions['Quantity'] * cleaned_transactions['Price']).round(8)).all())

# 2. Net revenue
chk('gross + return_value == net_revenue',
    abs((gross_revenue + return_value) - net_revenue) < 0.01,
    f'net = £{net_revenue:,.2f}')

# 3. Orders < rows
chk('n_orders < len(valid_merchandise_transactions)',
    n_orders < len(valid_merchandise_transactions),
    f'{n_orders:,} invoices vs {len(valid_merchandise_transactions):,} rows')

# 4. Monthly net sums to overall
chk('Monthly net_revenue sums to overall net',
    abs(monthly_kpis['net_revenue'].sum() - net_revenue) < 1.0,
    f'monthly sum = £{monthly_kpis["net_revenue"].sum():,.2f}')

# 5. Monthly orders sum
chk('Monthly orders sum to total orders',
    abs(monthly_kpis['orders'].sum() - n_orders) < 1,
    f'monthly sum = {int(monthly_kpis["orders"].sum()):,}')

# 6. Yearly net sums to overall
chk('Yearly net_revenue sums to overall net',
    abs(yearly_kpis['net_revenue'].sum() - net_revenue) < 1.0,
    f'yearly sum = £{yearly_kpis["net_revenue"].sum():,.2f}')

# 7. Customer revenue
chk('Customer-level revenue reconciles to customer_transactions',
    abs(customer_summary['total_revenue'].sum() -
        customer_transactions['revenue'].sum()) < 0.01)

# 8. Product gross revenue
chk('Product-level gross revenue reconciles to valid_merchandise_transactions',
    abs(product_summary['gross_revenue'].sum() -
        valid_merchandise_transactions['revenue'].sum()) < 0.01)

# 9. Country gross revenue
chk('Country-level gross revenue reconciles to valid_merchandise_transactions',
    abs(country_summary['gross_revenue'].sum() -
        valid_merchandise_transactions['revenue'].sum()) < 0.01)

print(f'\nAll {len(checks)} reconciliation checks passed.')

## 16. Section 14 — Executive KPI Findings

All values are derived from the dataset. No findings are fabricated or assumed.

In [ ]:
return_rate_pct  = return_value_abs / gross_revenue * 100
uk_row           = country_summary[country_summary['Country'] == 'United Kingdom']
uk_net           = uk_row['net_revenue'].values[0] if len(uk_row) > 0 else 0
uk_pct           = uk_net / net_revenue * 100
max_month_row    = monthly_kpis.loc[monthly_kpis['net_revenue'].idxmax()]
min_month_row    = monthly_kpis.loc[monthly_kpis['net_revenue'].idxmin()]
one_time_custs   = (customer_order_counts == 1).sum()
one_time_pct     = one_time_custs / total_id_customers * 100

# 2010 vs 2011 comparison (both full years)
net_2010 = yearly_kpis.loc[yearly_kpis['year'] == 2010, 'net_revenue'].values[0]
net_2011 = yearly_kpis.loc[yearly_kpis['year'] == 2011, 'net_revenue'].values[0]
yoy_2010_2011 = (net_2011 - net_2010) / net_2010 * 100

findings = [
    {
        'id': 1,
        'finding':
            f'Gross merchandise revenue across the observation period (Dec 2009 – Dec 2011) '
            f'is £{gross_revenue:,.0f}. Return and cancellation value totals '
            f'£{return_value_abs:,.0f} ({return_rate_pct:.1f}% of gross), '
            f'yielding net merchandise revenue of £{net_revenue:,.0f}.',
        'evidence':
            'gross_revenue: sum of revenue on valid_merchandise_transactions. '
            'return_value: sum of negative revenue on merchandise_product_transactions.',
        'business_relevance':
            'The gap between gross and net revenue quantifies the economic impact of returns. '
            'A return rate below 5% of gross is generally considered low for e-commerce.'
    },
    {
        'id': 2,
        'finding':
            f'{n_orders:,} unique merchandise purchase invoices were placed by '
            f'{n_customers:,} identified customers, with an average gross order '
            f'value of £{aov:,.2f} and {avg_items:.0f} items per order.',
        'evidence':
            'n_orders: Invoice.nunique() on valid_merchandise_transactions. '
            'aov: mean(invoice_revenue) on invoice_agg.',
        'business_relevance':
            'High average items per order (286) suggests bulk-buying behaviour '
            'consistent with a wholesale or B2B customer base rather than a typical '
            'B2C retail pattern.'
    },
    {
        'id': 3,
        'finding':
            f'{repeat_customers:,} of {total_id_customers:,} identified customers '
            f'({repeat_rate:.1f}%) made more than one purchase. '
            f'{one_time_custs:,} customers ({one_time_pct:.1f}%) made exactly one purchase.',
        'evidence':
            'customer_order_counts: Customer ID grouped by Invoice.nunique() '
            'on customer_transactions.',
        'business_relevance':
            'A 72% repeat customer rate is a strong retention signal. '
            'The 28% one-time-buyer segment represents the primary re-activation opportunity.'
    },
    {
        'id': 4,
        'finding':
            f'Comparing the two complete calendar years: '
            f'2010 net revenue = £{net_2010:,.0f}; '
            f'2011 net revenue = £{net_2011:,.0f} '
            f'({yoy_2010_2011:+.1f}% year-over-year). '
            f'Peak revenue month was {str(max_month_row["year_month"])} '
            f'(£{max_month_row["net_revenue"]:,.0f}); '
            f'lowest was {str(min_month_row["year_month"])} '
            f'(£{min_month_row["net_revenue"]:,.0f}).',
        'evidence':
            'yearly_kpis and monthly_kpis computed from merchandise_product_transactions.',
        'business_relevance':
            'The year-over-year decline in net revenue warrants further investigation '
            'into order volume trends, AOV movement, and return rate changes. '
            'Note that December 2011 data is partial (cut-off on 9 Dec), which '
            'depresses the 2011 full-year total.'
    },
    {
        'id': 5,
        'finding':
            f'United Kingdom accounts for £{uk_net:,.0f} net merchandise revenue '
            f'({uk_pct:.1f}% of all net revenue). '
            f'{len(country_summary)} countries are represented in total.',
        'evidence':
            'country_summary filtered to United Kingdom.',
        'business_relevance':
            'Heavy UK concentration means overall revenue is highly sensitive to '
            'UK-specific demand shocks. International markets represent '
            f'{100 - uk_pct:.1f}% of net revenue and may offer diversification opportunity.'
    },
]

for f in findings:
    print(f"Finding {f['id']}:")
    print(f"  Finding           : {f['finding']}")
    print(f"  Evidence          : {f['evidence']}")
    print(f"  Business relevance: {f['business_relevance']}")
    print()

## 17. Section 15 — Save Analytical Tables

Lightweight summary tables only — no full transaction dataset copies saved.

In [ ]:
# Convert Period to string before CSV serialization
monthly_kpis_save = monthly_kpis.copy()
monthly_kpis_save['year_month'] = monthly_kpis_save['year_month'].astype(str)

tables = {
    'customer_summary.csv'  : customer_summary,
    'product_summary.csv'   : product_summary,
    'country_summary.csv'   : country_summary,
    'monthly_kpis.csv'      : monthly_kpis_save,
    'yearly_kpis.csv'       : yearly_kpis,
}

for fname, df in tables.items():
    path = os.path.join(OUTPUTS_DIR, fname)
    df.to_csv(path, index=False)
    print(f'Saved: {path}  ({len(df):,} rows, {df.shape[1]} columns)')

print('\nTable purposes:')
print('  customer_summary  — one row per identified customer; input for RFM, segmentation, CLV')
print('  product_summary   — one row per StockCode; input for product intelligence')
print('  country_summary   — one row per country; input for geographic analytics')
print('  monthly_kpis      — one row per year-month; input for trend & cohort analysis')
print('  yearly_kpis       — one row per year; input for executive reporting')

---

## 18. Cleaning Summary & Decisions for Notebook 04

### What was built in this notebook

| Feature / Table | Description |
|-----------------|-------------|
| Time features | year, quarter, month, month_name, year_month, week, day, day_of_week, day_of_week_name, hour — on all 4 datasets |
| invoice_agg | 39,492 rows, one per merchandise invoice |
| Executive KPI table | 12 KPIs covering revenue, volume, customers, products, loyalty |
| monthly_kpis | 25 year-months, gross/net revenue, orders, units, customers, AOV, MoM growth |
| yearly_kpis | 3 years (2009 partial, 2010 full, 2011 full), YoY % changes |
| customer_summary | 5,851 customers, 8 metrics per customer |
| product_summary | 4,851 StockCodes, gross/net revenue, orders, customers |
| country_summary | 43 countries, gross/net revenue, AOV, customers, revenue per customer |

### Decisions required before Notebook 04

| # | Item | Decision needed |
|---|------|----------------|
| 1 | **2009 data (1 month only)** | Confirm whether to include or exclude December 2009 from trend/cohort analysis. Including it distorts any comparison that uses 2009 as a baseline year. |
| 2 | **December 2011 cut-off** | Data ends 9 Dec 2011. Confirm whether to include or truncate this partial month in cohort and retention analysis. |
| 3 | **AOV definition** | AOV is currently gross per invoice. Confirm whether net AOV (after returns) is preferred for downstream reporting. |
| 4 | **Unidentified customer rows** | 231,398 valid merchandise rows have no Customer ID (~22.7% of valid_merchandise_transactions). Confirm acceptable scope for market basket analysis — whether to include anonymous basket data or restrict to identified customers only. |
| 5 | **Reference date for RFM** | Recency requires a snapshot date. Confirm whether to use the dataset maximum date (2011-12-09) or a business-defined cutoff. |

*End of Notebook 03 — Feature Engineering & Executive KPIs.*